In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

In [3]:
key = {"district_key": str}

kba = pd.read_csv(CLEAN_DIR / "kba_stock.csv", dtype=key)
population = pd.read_csv(CLEAN_DIR / "population.csv", dtype=key)
income_res = pd.read_csv(CLEAN_DIR / "income_residence.csv", dtype=key)
income_work = pd.read_csv(CLEAN_DIR / "income_workplace.csv", dtype=key)
charging = pd.read_csv(CLEAN_DIR / "charging_points.csv", dtype=key)

for name, table in [("kba", kba), ("population", population), ("income_res", income_res),
                    ("income_work", income_work), ("charging", charging)]:
    print(f"{name:12} rows: {len(table):5}  first key: {table['district_key'].iloc[0]}")

kba          rows:  3996  first key: 08111
population   rows:  4005  first key: 01001
income_res   rows:  2005  first key: 01001
income_work  rows:  4010  first key: 01001
charging     rows:  3839  first key: 01002


In [4]:
KEY_MAP = {
    "16056": "16063",   # Eisenach -> Wartburgkreis (merged on 1 July 2021)
    "06415": "06435",   # Hanau -> Main-Kinzig-Kreis (separated on 1 January 2026)
    "07235": "07211",   # Trier-Saarburg -> Trier (one registration district at KBA)
}

for table in [kba, population, income_res, income_work, charging]:
    table["district_key"] = table["district_key"].replace(KEY_MAP)

for name, table in [("kba", kba), ("population", population), ("income_res", income_res),
                    ("income_work", income_work), ("charging", charging)]:
    print(f"{name:12} unique districts: {table['district_key'].nunique()}")

kba          unique districts: 399
population   unique districts: 399
income_res   unique districts: 399
income_work  unique districts: 399
charging     unique districts: 399


In [5]:
group = ["year", "district_key"]

kba_h = kba.groupby(group)[["cars_total", "bev"]].sum().reset_index()
population_h = population.groupby(group)["population"].sum().reset_index()
charging_h = charging.groupby(group)["charging_points"].sum().reset_index()
income_res_h = income_res.groupby(group)["income_median_residence"].mean().reset_index()
income_work_h = income_work.groupby(group)["income_median_workplace"].mean().reset_index()

for name, table in [("kba", kba_h), ("population", population_h), ("income_res", income_res_h),
                    ("income_work", income_work_h), ("charging", charging_h)]:
    print(f"{name:12}", table.groupby("year").size().to_dict())

kba          {2017: 399, 2018: 399, 2019: 399, 2020: 399, 2021: 399, 2022: 399, 2023: 399, 2024: 399, 2025: 399, 2026: 399}
population   {2017: 399, 2018: 399, 2019: 399, 2020: 399, 2021: 399, 2022: 399, 2023: 399, 2024: 399, 2025: 399, 2026: 399}
income_res   {2022: 399, 2023: 399, 2024: 399, 2025: 399, 2026: 399}
income_work  {2017: 399, 2018: 399, 2019: 399, 2020: 399, 2021: 399, 2022: 399, 2023: 399, 2024: 399, 2025: 399, 2026: 399}
charging     {2017: 285, 2018: 355, 2019: 388, 2020: 397, 2021: 399, 2022: 399, 2023: 399, 2024: 399, 2025: 399, 2026: 399}


In [6]:
panel = kba_h.merge(population_h, on=group, how="left")
panel = panel.merge(charging_h, on=group, how="left")
panel = panel.merge(income_res_h, on=group, how="left")
panel = panel.merge(income_work_h, on=group, how="left")

panel["charging_points"] = panel["charging_points"].fillna(0).astype(int)

print("Rows:", len(panel))
print(panel.isna().sum())
panel.head()

Rows: 3990
year                          0
district_key                  0
cars_total                    0
bev                           0
population                    0
charging_points               0
income_median_residence    1995
income_median_workplace       0
dtype: int64


,year,district_key,cars_total,bev,population,charging_points,income_median_residence,income_median_workplace
0,2017,01001,42321,42,87432,0,NaN,2933.983290
1,2017,01002,107982,64,247441,18,NaN,3231.205009
2,2017,01003,96030,62,216712,2,NaN,2961.929366
3,2017,01004,40776,23,79680,5,NaN,2760.435380
4,2017,01051,77783,47,133560,12,NaN,2834.875000


In [7]:
names = population[population["year"] == 2026][["district_key", "district_name"]]
names = names.drop_duplicates(subset="district_key")
panel = panel.merge(names, on="district_key", how="left")
panel.loc[panel["district_key"] == "07211", "district_name"] = "Trier + Trier-Saarburg"

STATES = {
    "01": "Schleswig-Holstein", "02": "Hamburg", "03": "Niedersachsen", "04": "Bremen",
    "05": "Nordrhein-Westfalen", "06": "Hessen", "07": "Rheinland-Pfalz", "08": "Baden-Württemberg",
    "09": "Bayern", "10": "Saarland", "11": "Berlin", "12": "Brandenburg",
    "13": "Mecklenburg-Vorpommern", "14": "Sachsen", "15": "Sachsen-Anhalt", "16": "Thüringen",
}
panel["state_key"] = panel["district_key"].str[:2]
panel["state"] = panel["state_key"].map(STATES)
panel["region"] = panel["state_key"].apply(lambda k: "East" if k >= "11" else "West")

print(panel["district_name"].isna().sum(), panel["state"].isna().sum())
panel.groupby(["year", "region"]).size().unstack()

0 0


region,East,West
year,,
2017,76,323
2018,76,323
2019,76,323
2020,76,323
2021,76,323
2022,76,323
2023,76,323
2024,76,323
2025,76,323


In [8]:
panel["bev_share_pct"] = panel["bev"] / panel["cars_total"] * 100
panel["cars_per_1000_inh"] = panel["cars_total"] / panel["population"] * 1000
panel["charging_points_per_100k_inh"] = panel["charging_points"] / panel["population"] * 100_000
panel["bev_per_charging_point"] = panel["bev"] / panel["charging_points"]

panel[["bev_share_pct", "cars_per_1000_inh", "charging_points_per_100k_inh", "bev_per_charging_point"]].describe().round(1)

/home/sc4r3cr0w/anaconda3/lib/python3.14/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,bev_share_pct,cars_per_1000_inh,charging_points_per_100k_inh,bev_per_charging_point
count,3990.0,3990.0,3990.0,3990.0
mean,1.4,606.2,82.9,inf
std,1.5,76.7,96.2,NaN
min,0.0,330.4,0.0,0.7
25%,0.2,568.9,14.0,6.5
50%,0.8,619.9,47.7,10.3
75%,2.4,655.7,125.4,15.3
max,12.5,1129.8,1066.8,inf


In [9]:
panel["bev_per_charging_point"] = panel["bev_per_charging_point"].replace(np.inf, np.nan)

print("Missing values:", panel["bev_per_charging_point"].isna().sum())
panel[["bev_share_pct", "cars_per_1000_inh", "charging_points_per_100k_inh", "bev_per_charging_point"]].describe().round(1)

Missing values: 171


,bev_share_pct,cars_per_1000_inh,charging_points_per_100k_inh,bev_per_charging_point
count,3990.0,3990.0,3990.0,3819.0
mean,1.4,606.2,82.9,12.1
std,1.5,76.7,96.2,12.6
min,0.0,330.4,0.0,0.7
25%,0.2,568.9,14.0,6.4
50%,0.8,619.9,47.7,10.0
75%,2.4,655.7,125.4,14.3
max,12.5,1129.8,1066.8,407.0


In [10]:
columns = [
    "year", "district_key", "district_name", "state_key", "state", "region",
    "population", "cars_total", "bev", "charging_points",
    "income_median_residence", "income_median_workplace",
    "bev_share_pct", "cars_per_1000_inh", "charging_points_per_100k_inh", "bev_per_charging_point",
]
panel = panel[columns].sort_values(["district_key", "year"]).reset_index(drop=True)

panel.to_csv(CLEAN_DIR / "panel_districts.csv", index=False)
print("Saved:", CLEAN_DIR / "panel_districts.csv", "| rows:", len(panel), "| columns:", len(panel.columns))
panel.head()

Saved: /home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/CLEAN/panel_districts.csv | rows: 3990 | columns: 16


,year,district_key,district_name,state_key,state,region,population,cars_total,bev,charging_points,income_median_residence,income_median_workplace,bev_share_pct,cars_per_1000_inh,charging_points_per_100k_inh,bev_per_charging_point
0,2017,01001,"Flensburg, kreisfreie Stadt",01,Schleswig-Holstein,West,87432,42321,42,0,NaN,2933.983290,0.099242,484.044743,0.000000,NaN
1,2018,01001,"Flensburg, kreisfreie Stadt",01,Schleswig-Holstein,West,88519,43075,64,6,NaN,2986.055556,0.148578,486.618692,6.778206,10.666667
2,2019,01001,"Flensburg, kreisfreie Stadt",01,Schleswig-Holstein,West,89504,43822,94,7,NaN,3102.117251,0.214504,489.609403,7.820880,13.428571
3,2020,01001,"Flensburg, kreisfreie Stadt",01,Schleswig-Holstein,West,90164,44556,144,20,NaN,3177.137280,0.323189,494.166186,22.181802,7.200000
4,2021,01001,"Flensburg, kreisfreie Stadt",01,Schleswig-Holstein,West,89934,45029,321,30,NaN,3190.171053,0.712874,500.689394,33.357796,10.700000
